# Super-Convergence: ResNet-20 / CIFAR-10

1. Runtime → Change runtime type → T4 GPU.
2. Set `REPO_URL` in the Config cell.
3. Runtime → Run all.

After a disconnect or a runtime restart, Run all again: finished runs are skipped, interrupted ones resume. Do not change `SCALE` once the queue has started.

## Config

In [ ]:
REPO_URL = 'https://github.com/bashmikh/super-convergence-repro.git'
REPO = '/content/repo'                     # clone location
OUT = '/content/drive/MyDrive/sc-repro'    # results, checkpoints, TensorBoard
DATA = '/content/data'                     # CIFAR-10 (local disk, faster than Drive)
SCALE = 1.0                                # iteration budget multiplier for run_all

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
assert '<user>' not in REPO_URL, 'Set REPO_URL in the Config cell'
!test -d {REPO}/.git || git clone -q {REPO_URL} {REPO}
!git -C {REPO} pull -q
!pip install -q uv && uv pip install --system -q -e {REPO}
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Environment check and smoke test

In [ ]:
!python {REPO}/scripts/doctor.py --out {OUT} --data {DATA}
!python {REPO}/scripts/smoke.py --out /content/smoke --scale {SCALE}

## TensorBoard

Keeps updating while the experiment cell runs.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUT}/tb

## Experiments

Add `--only E1,E3` to restrict the queue.

In [ ]:
!python {REPO}/scripts/run_all.py --out {OUT} --data {DATA} --scale {SCALE}

## Figures and table

In [ ]:
!python {REPO}/scripts/plots.py --out {OUT} --readme {REPO}/README.md
from pathlib import Path
from IPython.display import Image, Markdown, display
for p in sorted(Path(OUT, 'figures').glob('*.png')):
    display(Image(str(p)))
display(Markdown(Path(OUT, 'summary.md').read_text()))